## Reading Annotations

In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

human_annotations_json = '../data/2024-01-16-Count-People-Stories-Human-Annotations.json' # @param {type:"string"}

with open(human_annotations_json, 'r') as f:
  j = f.read()
  exported_annotations = json.loads(j)

def process_result(result, coder, md):
    value_type = result.get('type', "")
    metadata = {
        **md,
        "coder": coder,
        "from_name": result.get('from_name', "")
    }
    annotations = []
    if value_type == "choices":
        choices = result['value'].get('choices', [])
        for choice in choices:
            r = {**metadata}
            r['value'] = choice
            annotations.append(r)
    elif value_type == "taxonomy":
        taxonomies = result['value'].get('taxonomy', [])
        for taxonomy in taxonomies:
            if len(taxonomy) > 1:
                taxonomy = " > ".join(taxonomy)
            elif len(taxonomy) == 1:
                taxonomy = taxonomy[0]
            r = {**metadata}
            r['value'] = taxonomy
            annotations.append(r)
    return annotations

all_annotations = []

for data in tqdm(exported_annotations):
    annotations = data.get("annotations")
    metadata = {
        **data.get("data")
    }

    for annotation in annotations:
      coder = annotation['completed_by']['id']
      results = annotation.get("result")
      if results:
        for result in results:
          all_annotations.extend(process_result(result, coder, metadata))
      else:
        print("Skipped Missing Result")


all_annotations_df = pd.DataFrame(all_annotations)

  0%|          | 0/281 [00:00<?, ?it/s]

In [ ]:
all_annotations_df

,Image,coder,from_name,value
0,gs://ig-politik-stories/abaerbock_26-09-2021_0...,10475,Crowd,No
1,gs://ig-politik-stories/abaerbock_26-09-2021_0...,10475,PersonCount,OnePerson
2,gs://ig-politik-stories/abaerbock_26-09-2021_0...,16195,Crowd,No
3,gs://ig-politik-stories/abaerbock_26-09-2021_0...,16195,PersonCount,OnePerson
4,gs://ig-politik-stories/abaerbock_26-09-2021_0...,7107,Crowd,No
...,...,...,...,...
1681,gs://ig-politik-stories/robert.habeck_22-09-20...,10475,PersonCount,OnePerson
1682,gs://ig-politik-stories/robert.habeck_22-09-20...,16195,Crowd,No
1683,gs://ig-politik-stories/robert.habeck_22-09-20...,16195,PersonCount,OnePerson
1684,gs://ig-politik-stories/robert.habeck_22-09-20...,7107,Crowd,No


In [ ]:
all_annotations_df["Image"] = all_annotations_df["Image"].apply(lambda x: x.replace("../data/", ""))

In [ ]:
from_name = 'PersonCount' # @param {type:"string"}
identifier = 'Image' # @param {type:"string"}

# Assuming your DataFrame is named 'df'
filtered_df = all_annotations_df.copy()
filtered_df = filtered_df[filtered_df['from_name'].str.contains(from_name, case=False)]

def to_bool(val):
  if isinstance(val, bool):
    return val
  if isinstance(val, str):
    return val.lower() == "true"

  return bool(val)

def ls_face_count(n):
    if n == "OnePerson":
      return str(1)
    elif n == "TwoPeople":
      return str(2)
    elif n == "ThreeToFivePeople":
      return "3+"
    elif n == "SixOrMore":
      return "3+"
    else:
      return "0"

filtered_df['value'] = filtered_df['value'].apply(ls_face_count)
values = filtered_df['value'].unique()
identifier_values = filtered_df[identifier].unique()

contingency_matrix = pd.crosstab(filtered_df[identifier], filtered_df['coder'], values=filtered_df['value'], aggfunc='first')
contingency_matrix = contingency_matrix.reindex(identifier_values)

In [ ]:
contingency_matrix

coder,7107,10475,16195
Image,,,
abaerbock_26-09-2021_00:00_video0.jpeg,1,1,1
robert.habeck_17-09-2021_00:00_image11.jpeg,3+,3+,3+
armin_laschet_21-09-2021_00:00_video1.jpeg,1,1,1
abaerbock_25-09-2021_00:00_video0.jpeg,3+,3+,3+
olafscholz_24-09-2021_00:00_image2.jpeg,2,2,2
...,...,...,...
robert.habeck_21-09-2021_00:00_video8.jpeg,1,1,1
armin_laschet_22-09-2021_00:00_video11.jpeg,3+,2,2
christianlindner_18-09-2021_00:00_video1.jpeg,3+,2,2


Export list of annotations to save money with GPT

In [ ]:
contingency_matrix.to_csv('../data/2024-07-10-Story-Person-Count-Annotation.csv')

In [ ]:
contingency_matrix = pd.read_csv('../data/2024-07-10-Story-Person-Count-Annotation.csv')

In [ ]:
contingency_matrix.head()

,Image,7107,10475,16195
0,abaerbock_26-09-2021_00:00_video0.jpeg,1,1,1
1,robert.habeck_17-09-2021_00:00_image11.jpeg,3+,3+,3+
2,armin_laschet_21-09-2021_00:00_video1.jpeg,1,1,1
3,abaerbock_25-09-2021_00:00_video0.jpeg,3+,3+,3+
4,olafscholz_24-09-2021_00:00_image2.jpeg,2,2,2


## Interrater Agreement

In [ ]:
!pip install -q krippendorff

In [ ]:
import krippendorff
import pandas as pd

def convert_to_reliability_data(matrix):
    transposed_matrix = matrix.T
    reliability_data = []
    for _, ratings in transposed_matrix.iterrows():
        reliability_data.append(ratings.tolist())
    return reliability_data

test_data = contingency_matrix[["7107", "10475", "16195"]]  # Nur Katharina und Michael
reliability_data = convert_to_reliability_data(test_data)   # contingency_matrix)

# Calculating Krippendorff's Alpha treating "Unsure" as a distinct category
alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

print("Krippendorff's Alpha:", alpha)

Krippendorff's Alpha: 0.8078056091510474


## Majority Decision

In [ ]:
def get_majority_vote(row):
    # Count the votes
    votes = row.dropna().tolist()
    vote_counts = pd.Series(votes).value_counts()

    # Check if there's a clear majority
    if len(vote_counts) == 1 or vote_counts.iloc[0] > vote_counts.iloc[1]:
        return vote_counts.index[0]
    else:
        # If tie or only two votes, use column 7107
        return row[7107]

# Apply the function to each row
contingency_matrix['Gold'] = contingency_matrix.apply(get_majority_vote, axis=1)

In [ ]:
contingency_matrix = contingency_matrix.reset_index()

In [ ]:
contingency_matrix

,index,Image,7107,10475,16195,Gold
0,0,abaerbock_26-09-2021_00:00_video0.jpeg,1,1,1,1
1,1,robert.habeck_17-09-2021_00:00_image11.jpeg,3+,3+,3+,3+
2,2,armin_laschet_21-09-2021_00:00_video1.jpeg,1,1,1,1
3,3,abaerbock_25-09-2021_00:00_video0.jpeg,3+,3+,3+,3+
4,4,olafscholz_24-09-2021_00:00_image2.jpeg,2,2,2,2
...,...,...,...,...,...,...
276,276,robert.habeck_21-09-2021_00:00_video8.jpeg,1,1,1,1
277,277,armin_laschet_22-09-2021_00:00_video11.jpeg,3+,2,2,2
278,278,christianlindner_18-09-2021_00:00_video1.jpeg,3+,2,2,2
279,279,olafscholz_20-09-2021_00:00_video4.jpeg,1,1,1,1


In [ ]:
#@title Qualitative Inspection --- Human Annotations
#@markdown Run this cell to generate a table comparing the Human and Model classifications.
images_per_label = 7  # @param {type: "slider", min: 1, max: 200}

import json
from google.cloud import storage
from google.oauth2 import service_account
from datetime import timedelta, datetime
import pandas as pd
from IPython.display import display, HTML

# Group the DataFrame by 'Majority Decision'
filtered = contingency_matrix.copy()

# Load the credentials from the JSON file
with open("# REMOVED: GCP service account key not needed for local execution", 'r') as f:
    credentials_info = json.load(f)

# Create credentials object from the service account information
credentials = service_account.Credentials.from_service_account_info(credentials_info)

# Create a storage client object using the credentials
storage_client = storage.Client(credentials=credentials)

# Set the expiration time to 7 days from now
expiration_time = timedelta(days=7)

# Iterate through the rows in the DataFrame and sign the URLs
for index, row in filtered.iterrows():
    # Split the gcloud URL to get bucket name and blob name
    bucket_name = "ig-politik-stories"
    blob_name = row['Image']

    # Get the bucket and blob objects
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    # Generate the signed URL, passing the expiration time
    url = blob.generate_signed_url(expiration=datetime.utcnow() + expiration_time)
    filtered.loc[index, 'gcloud'] = url

grouped = filtered.groupby('Gold')

# HTML template with Bootstrap
html_template = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
    <title>Grouped Images</title>
</head>
<body>
    <div class="container">
        {content}
    </div>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
'''

# Generating the content for each group
content = ''
for decision, group in grouped:
    if images_per_label >= len(group):
      sample_size = len(group)
    else:
      sample_size = images_per_label

    sample = group.sample(sample_size)
    content += f'<h2>{decision}</h2>'
    content += '<div class="row">'
    for _, row in sample.iterrows():
        content += f'''
        <div class="col-md-4">
            <div class="card mb-4 shadow-sm">
                <img src="{row['gcloud']}" class="card-img-top" alt="{row['Image']}">
                <div class="card-body">
                    <p class="card-text">{row['Image']}</p>
                    <p class="card-text">Humans: {row['Gold']}</p>
                </div>
            </div>
        </div>
        '''
    content += '</div>'

# Combine the content with the HTML template
html_output = html_template.format(content=content)

# Write the HTML to a file
with open('grouped_images_facenet.html', 'w') as file:
    file.write(html_output)

print("HTML file 'grouped_images_facenet.html' has been created.")

# Function to display the HTML in Colab
def display_html(file_path):
    with open(file_path, 'r') as file:
        html_content = file.read()
    display(HTML(html_content))

# Display the HTML file in the notebook
display_html('grouped_images_facenet.html')

## Evaluation GPT

In [ ]:
gpt_df = pd.read_csv('../data/2024-07-04-Stories-Count-Annotation-Not-Sampled-GPT4o-V3-TEST.csv')

In [ ]:
contingency_table = pd.merge(contingency_matrix, gpt_df[['Image', "GPT Count"]], on="Image", how='right')
contingency_table.rename(columns={"GPT Count": "GPT-4o-V3"}, inplace=True)

In [ ]:
import re

# Convert entire columns to string type
contingency_table = contingency_table.copy()
contingency_table['GPT-4o-V3'] = contingency_table['GPT-4o-V3'].astype(str)

def repl(input):
  input = input.strip()
  if input == '0':
    return '0'
  if input == '1':
    return '1'
  elif input == '2':
    return '2'
  elif input == '3' or input == '3+':
    return '3+'
  elif pd.isna(input):
    return "Problem"
  elif int(input) > 3:
    return '3+'
  else:
    return "Problem"

# Apply the regex substitution
contingency_table['GPT-4o-V3'] = contingency_table['GPT-4o-V3'].apply(repl)

In [ ]:
contingency_table.head()

,index,Image,7107,10475,16195,Gold,GPT-4o-V3
0,NaN,abaerbock_13-09-2021_00:00_image0.jpeg,NaN,NaN,NaN,NaN,1
1,NaN,abaerbock_13-09-2021_00:00_image2.jpeg,NaN,NaN,NaN,NaN,1
2,NaN,abaerbock_13-09-2021_00:00_image3.jpeg,NaN,NaN,NaN,NaN,1
3,NaN,abaerbock_13-09-2021_00:00_image4.jpeg,NaN,NaN,NaN,NaN,1
4,NaN,abaerbock_13-09-2021_00:00_image5.jpeg,NaN,NaN,NaN,NaN,1


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from IPython.display import display, Markdown

# Calculating overall metrics
accuracy = accuracy_score(contingency_table['Gold'], contingency_table['GPT-4o-V3'])
precision = precision_score(contingency_table['Gold'], contingency_table['GPT-4o-V3'], average='macro')
recall = recall_score(contingency_table['Gold'], contingency_table['GPT-4o-V3'], average='macro')
f1 = f1_score(contingency_table['Gold'], contingency_table['GPT-4o-V3'], average='macro')

# Creating a DataFrame for the overall metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'GPT': [accuracy, precision, recall, f1]
})

# Displaying the DataFrame as a table
display(Markdown("### Overall Metrics"))
display(metrics_df)

# Generating the classification report for each label
# filter = contingency_table[contingency_table['Gold'] != "0"]
report = classification_report(contingency_table['Gold'], contingency_table['GPT-4o-V3'], output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Displaying the classification report as a table
display(Markdown("### Classification Report for Each Label"))
display(report_df)


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Generate the confusion matrix
cm = confusion_matrix(contingency_table['Gold'], contingency_table['GPT-4o-V3'])

# Plotting the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(0), str(1), str(2), "3+"], yticklabels=[str(0), str(1), str(2), "3+"])
plt.title('GPT-4o')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import krippendorff
import pandas as pd

def convert_to_reliability_data(matrix):
    transposed_matrix = matrix.T
    reliability_data = []
    for _, ratings in transposed_matrix.iterrows():
        reliability_data.append(ratings.tolist())
    return reliability_data

test_data = contingency_table[["7107", "10475", "16195", 'GPT-4o-V3']]  # Nur Katharina und Michael
reliability_data = convert_to_reliability_data(test_data)   # contingency_matrix)

# Calculating Krippendorff's Alpha treating "Unsure" as a distinct category
alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

print("Krippendorff's Alpha mit GPT-4o-V3:", alpha)

In [ ]:
#@title Qualitative Inspection
#@markdown Run this cell to generate a table comparing the Human and Model classifications.
images_per_label = 3  # @param {type: "slider", min: 1, max: 200}

import json
from google.cloud import storage
from google.oauth2 import service_account
from datetime import timedelta, datetime
import pandas as pd
from IPython.display import display, HTML

# Group the DataFrame by 'Majority Decision'
filtered = contingency_table[contingency_table['Gold'] != contingency_table['GPT-4o-V3']].copy()

# Load the credentials from the JSON file
with open("# REMOVED: GCP service account key not needed for local execution", 'r') as f:
    credentials_info = json.load(f)

# Create credentials object from the service account information
credentials = service_account.Credentials.from_service_account_info(credentials_info)

# Create a storage client object using the credentials
storage_client = storage.Client(credentials=credentials)

# Set the expiration time to 7 days from now
expiration_time = timedelta(days=7)

# Iterate through the rows in the DataFrame and sign the URLs
for index, row in filtered.iterrows():
    # Split the gcloud URL to get bucket name and blob name
    bucket_name = "ig-politik-stories"
    blob_name = row['Image']

    # Get the bucket and blob objects
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    # Generate the signed URL, passing the expiration time
    url = blob.generate_signed_url(expiration=datetime.utcnow() + expiration_time)
    filtered.loc[index, 'gcloud'] = url

grouped = filtered.groupby('Gold')

# HTML template with Bootstrap
html_template = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
    <title>Grouped Images</title>
</head>
<body>
    <div class="container">
        {content}
    </div>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
'''

# Generating the content for each group
content = ''
for decision, group in grouped:
    if images_per_label >= len(group):
      sample_size = len(group)
    else:
      sample_size = images_per_label

    sample = group.sample(sample_size)
    content += f'<h2>{decision}</h2>'
    content += '<div class="row">'
    for _, row in sample.iterrows():
        content += f'''
        <div class="col-md-4">
            <div class="card mb-4 shadow-sm">
                <img src="{row['gcloud']}" class="card-img-top" alt="{row['Image']}">
                <div class="card-body">
                    <p class="card-text">{row['Image']}</p>
                    <p class="card-text">GPT-4o: {row['GPT-4o-V3']}</p>
                    <p class="card-text">Humans: {row['Gold']}</p>
                </div>
            </div>
        </div>
        '''
    content += '</div>'

# Combine the content with the HTML template
html_output = html_template.format(content=content)

# Write the HTML to a file
with open('grouped_images_facenet.html', 'w') as file:
    file.write(html_output)

print("HTML file 'grouped_images_facenet.html' has been created.")

# Function to display the HTML in Colab
def display_html(file_path):
    with open(file_path, 'r') as file:
        html_content = file.read()
    display(HTML(html_content))

# Display the HTML file in the notebook
display_html('grouped_images_facenet.html')

## Evaluation GPT w/o Crowd

In [ ]:
gpt_wo_df = pd.read_csv('../data/2024-07-09-Post-Count-Annotation-Not-Sampled-GPT4o-V3-wo-crowd.csv')

In [ ]:
contingency_table = pd.merge(contingency_matrix, gpt_wo_df[['Image', "GPT Count"]], left_on="filename", right_on="Image", how='left')
contingency_table.rename(columns={"GPT Count": "GPT-4o-V3-wo"}, inplace=True)

In [ ]:
import re

# Convert entire columns to string type
contingency_table = contingency_table.copy()
contingency_table['GPT-4o-V3-wo'] = contingency_table['GPT-4o-V3-wo'].astype(str)

def repl(input):
  input = input.strip()
  if input == '0':
    return '0'
  if input == '1':
    return '1'
  elif input == '2':
    return '2'
  elif input == '3' or input == '3+':
    return '3+'
  elif int(input) > 3:
    return '3+'
  else:
    return "Problem"

# Apply the regex substitution
contingency_table['GPT-4o-V3-wo'] = contingency_table['GPT-4o-V3-wo'].apply(repl)

In [ ]:
contingency_table.head()

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from IPython.display import display, Markdown

# Calculating overall metrics
accuracy = accuracy_score(contingency_table['Gold'], contingency_table['GPT-4o-V3-wo'])
precision = precision_score(contingency_table['Gold'], contingency_table['GPT-4o-V3-wo'], average='macro')
recall = recall_score(contingency_table['Gold'], contingency_table['GPT-4o-V3-wo'], average='macro')
f1 = f1_score(contingency_table['Gold'], contingency_table['GPT-4o-V3-wo'], average='macro')

# Creating a DataFrame for the overall metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'GPT': [accuracy, precision, recall, f1]
})

# Displaying the DataFrame as a table
display(Markdown("### Overall Metrics"))
display(metrics_df)

# Generating the classification report for each label
# filter = contingency_table[contingency_table['Gold'] != "0"]
report = classification_report(contingency_table['Gold'], contingency_table['GPT-4o-V3-wo'], output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Displaying the classification report as a table
display(Markdown("### Classification Report for Each Label"))
display(report_df)


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Generate the confusion matrix
cm = confusion_matrix(contingency_table['Gold'], contingency_table['GPT-4o-V3-wo'])

# Plotting the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(0), str(1), str(2), "3+"], yticklabels=[str(0), str(1), str(2), "3+"])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## Evaluation Deepface

In [ ]:
deepface_df = pd.read_csv('../data/AllFaces_v2.csv')

In [ ]:
head_count = deepface_df['filename'].value_counts().reset_index()

In [ ]:
head_count.head()

In [ ]:
contingency_table = pd.merge(contingency_table, head_count[['filename', "count"]], left_on="Image", right_on="filename", how='left')
contingency_table.rename(columns={"count": "deepface"}, inplace=True)

In [ ]:
import re

# Convert entire columns to string type
contingency_table = contingency_table.copy()
contingency_table['deepface'] = contingency_table['deepface'].astype(str)

def repl(input):
  input = input.strip()
  if input == '0':
    return '0'
  if input == '1':
    return '1'
  elif input == '2':
    return '2'
  elif input == '3' or input == '3+':
    return '3+'
  elif int(input) > 3:
    return '3+'
  else:
    return "Problem"

# Apply the regex substitution
contingency_table['deepface'] = contingency_table['deepface'].apply(repl)

In [ ]:
contingency_table.head()

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from IPython.display import display, Markdown

# Calculating overall metrics
accuracy = accuracy_score(contingency_table['Gold'], contingency_table['deepface'])
precision = precision_score(contingency_table['Gold'], contingency_table['deepface'], average='macro')
recall = recall_score(contingency_table['Gold'], contingency_table['deepface'], average='macro')
f1 = f1_score(contingency_table['Gold'], contingency_table['deepface'], average='macro')

# Creating a DataFrame for the overall metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'GPT': [accuracy, precision, recall, f1]
})

# Displaying the DataFrame as a table
display(Markdown("### Overall Metrics"))
display(metrics_df)

# Generating the classification report for each label
# filter = contingency_table[contingency_table['Gold'] != "0"]
report = classification_report(contingency_table['Gold'], contingency_table['deepface'], output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Displaying the classification report as a table
display(Markdown("### Classification Report for Each Label"))
display(report_df)


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Generate the confusion matrix
cm = confusion_matrix(contingency_table['Gold'], contingency_table['deepface'])

# Plotting the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(0), str(1), str(2), "3+"], yticklabels=[str(0), str(1), str(2), "3+"])
plt.title('RetinaFace')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import krippendorff
import pandas as pd

def convert_to_reliability_data(matrix):
    transposed_matrix = matrix.T
    reliability_data = []
    for _, ratings in transposed_matrix.iterrows():
        reliability_data.append(ratings.tolist())
    return reliability_data

test_data = contingency_table[[7107, 10475, 16195, 'deepface']]
reliability_data = convert_to_reliability_data(test_data)   # contingency_matrix)

# Calculating Krippendorff's Alpha treating "Unsure" as a distinct category
alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

print("Krippendorff's Alpha mit GCV:", alpha)

## Evaluation Google Cloud Vision

In [ ]:
gcv_df = pd.read_csv('../data/2023-10-02-Story-Google-Cloud-Vision-Object-Detection.csv')

In [ ]:
contingency_table = contingency_matrix[contingency_matrix['Image'].isin(gpt_df['Image'])].copy()

In [ ]:
def count_people(row):
    o = gcv_df[gcv_df['image'] == row['Image']]
    o = o[o['object_name'] == "Person"]
    # Get value counts of object_name
    object_counts = o['object_name'].value_counts()

    if len(object_counts) == 1:
      return object_counts[0]
    else:
      return 0

contingency_table['GCV'] = contingency_table.apply(count_people, axis=1)

In [ ]:
def int_to_str(n):
  if n < 3:
    return str(n)
  elif n >= 3:
    return "3+"
  else:
    return "Problem"

In [ ]:
contingency_table['GCV'] = contingency_table['GCV'].apply(int_to_str)

In [ ]:
contingency_table.head()

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from IPython.display import display, Markdown

# Calculating overall metrics
accuracy = accuracy_score(contingency_table['Gold'], contingency_table['GCV'])
precision = precision_score(contingency_table['Gold'], contingency_table['GCV'], average='macro')
recall = recall_score(contingency_table['Gold'], contingency_table['GCV'], average='macro')
f1 = f1_score(contingency_table['Gold'], contingency_table['GCV'], average='macro')

# Creating a DataFrame for the overall metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'GPT': [accuracy, precision, recall, f1]
})

# Displaying the DataFrame as a table
display(Markdown("### Overall Metrics"))
display(metrics_df)

# Generating the classification report for each label
# filter = contingency_table[contingency_table['Gold'] != "0"]
report = classification_report(contingency_table['Gold'], contingency_table['GCV'], output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Displaying the classification report as a table
display(Markdown("### Classification Report for Each Label"))
display(report_df)


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Generate the confusion matrix
cm = confusion_matrix(contingency_table['Gold'], contingency_table['GCV'])

# Plotting the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(0), str(1), str(2), "3+"], yticklabels=[str(0), str(1), str(2), "3+"])
plt.title('Google Cloud Vision')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import krippendorff
import pandas as pd

def convert_to_reliability_data(matrix):
    transposed_matrix = matrix.T
    reliability_data = []
    for _, ratings in transposed_matrix.iterrows():
        reliability_data.append(ratings.tolist())
    return reliability_data

test_data = contingency_table[[7107, 10475, 16195, 'GCV']]
reliability_data = convert_to_reliability_data(test_data)   # contingency_matrix)

# Calculating Krippendorff's Alpha treating "Unsure" as a distinct category
alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

print("Krippendorff's Alpha mit GCV:", alpha)